In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/mSHIFT_SHeS/code_ocean/

/content/drive/MyDrive/mSHIFT_SHeS/code_ocean


The notebook contains the scripts to produce the supplementary tables

In [ ]:
import pandas as pd
import numpy as np
import json

from pathlib import Path
import sys
from rpy2.robjects.packages import importr

data_path = Path("data")
results_path = Path("results")

In [ ]:
# Add the parent directory of the notebook to sys.path, enables module imports from notebook_code
sys.path.append(str(Path().resolve() / 'code/notebooks/notebook_code'))

In [ ]:
sys.path.append(str(Path().resolve() / 'code/mSHIFT'))

In [ ]:
import results_tables
from results_tables import results_table

In [ ]:
# # Load the conditions for picking out different demographic groups

# with open(data_path / 'demographic_groups/overall_group.pkl', 'rb') as fp:
#   overall_group = dill.load(fp)

# with open(data_path / 'demographic_groups/age_groups.pkl', 'rb') as fp:
#   age_group = dill.load(fp)

# with open(data_path / 'demographic_groups/sex_group.pkl', 'rb') as fp:
#   sex_group = dill.load(fp)

# with open(data_path / 'demographic_groups/simd_group.pkl', 'rb') as fp:
#   simd_group = dill.load(fp)

# demographic_groups = [overall_group, age_group, sex_group, simd_group]

In [ ]:
import dill

In [ ]:
# with open(data_path / "mappings/scenario_label_dictionary.pkl", "rb") as fp:
#   scenario_label_dictionary = dill.load(fp)

with open(data_path / 'mappings/nutrient_label_dict.json', 'r') as fp:
    nutrient_label_dict = json.load(fp)

In [ ]:
scenario_label_map ={22: 'CCC 2030',
 23: 'CCC 2050',
 1: 'SDG',
 2: 'Max red meat 60g/day',
 3: 'Max red meat 31g/day',
 34: "20% dairy reduction",
 35: "20% dairy reduction, plant-based dairy alternatives",
 24: 'CCC 2030, pulses & legumes',
 25: 'CCC 2050, pulses & legumes',
 4: 'SDG, pulses & legumes',
 5: 'Max red meat 60g/day, pulses & legumes',
 6: 'Max red meat 31g/day, pulses & legumes',
 26: 'CCC 2030, vegetables',
 27: 'CCC 2050, vegetables',
 7: 'SDG, vegetables',
 8: 'Max red meat 60g/day, vegetables',
 9: 'Max red meat 31g/day, vegetables',
 28: 'CCC 2030, egg',
 29: 'CCC 2050, egg',
 10: 'SDG, egg',
 11: 'Max red meat 60g/day, egg',
 12: 'Max red meat 31g/day, egg',
 30: 'CCC 2030, oily fish',
 31: 'CCC 2050, oily fish',
 13: 'SDG, oily fish',
 14: 'Max red meat 60g/day, oily fish',
 15: 'Max red meat 31g/day, oily fish',
 32: 'CCC 2030, plant-based meat alternatives',
 33: 'CCC 2050, plant-based meat alternatives',
 16: 'SDG, plant-based meat alternatives',
 17: 'Max red meat 60g/day, plant-based meat alternatives',
 18: 'Max red meat 31g/day, plant-based meat alternatives',
 19: 'SDG, poultry',
 20: 'Max red meat 60g/day, poultry',
 21: 'Max red meat 31g/day, poultry'}

scenario_label_map = {f"Scenario_{index}": value for index, value in scenario_label_map.items()}

In [ ]:
df_baseline = pd.read_parquet(data_path / 'df_baseline.parquet')
nutrients = np.loadtxt(data_path / 'indicator_lists/nutrients.txt', dtype=str).tolist()
env_columns = np.loadtxt(data_path / 'indicator_lists/env_columns.txt', dtype=str).tolist()
mean_env_columns = np.loadtxt(data_path / 'indicator_lists/mean_env_columns.txt', dtype=str).tolist()

all_indicators = nutrients + env_columns
error_columns = np.loadtxt(data_path / 'indicator_lists/error_columns.txt', dtype=str).tolist()

In [ ]:
overall_group = [("Overall", [ {'column': None, 'operator': None, 'value': None, 'boolean_operator': None}])]

subgroup_list_age = [

  ("16-24", [
                    {'column': 'age', 'operator': '>=', 'value': 16, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 24, 'boolean_operator': None}
                ]),

   ("25-34", [
      {'column': 'age', 'operator': '>=', 'value': 25, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 34, 'boolean_operator': None}
  ]),

   ("35-44", [
      {'column': 'age', 'operator': '>=', 'value': 35, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 44, 'boolean_operator': None}
  ]),

  ("45-54", [
      {'column': 'age', 'operator': '>=', 'value': 45, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 54, 'boolean_operator': None}
  ]),

  ("55-64", [
      {'column': 'age', 'operator': '>=', 'value': 55, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 64, 'boolean_operator': None}
  ]),

  ("65-74", [
      {'column': 'age', 'operator': '>=', 'value': 65, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 74, 'boolean_operator': None}
  ]),

  ("75+", [
      {'column': 'age', 'operator': '>=', 'value': 75, 'boolean_operator': None},

  ]),
            ]

subgroup_list_sex = [
  ("Female", [{'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("Male", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': None}])
]

subgroup_list_simd = [
  ("SIMD 1 (most deprived)", [{'column': 'SIMD1', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("SIMD 2", [{'column': 'SIMD2', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("SIMD 3", [{'column': 'SIMD3', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("SIMD 4", [{'column': 'SIMD4', 'operator': '==', 'value': 1, 'boolean_operator': None}]),
  ("SIMD 5 (least deprived)", [{'column': 'SIMD5', 'operator': '==', 'value': 1, 'boolean_operator': None}]),

]

subgroup_list_age_sex = [

                  ############## Male #####################

  ("M 16-24", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 16, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 24, 'boolean_operator': None}
                ]),

  ("M 25-34", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 25, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 34, 'boolean_operator': None}
                ]),

   ("M 35-44", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 35, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 44, 'boolean_operator': None}
  ]),

    ("M 45-54", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 45, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 54, 'boolean_operator': None}
  ]),

                  ("M 55-64", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 55, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 64, 'boolean_operator': None}
  ]),

  ("M 65+", [{'column': 'Sex', 'operator': '==', 'value': 0, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 65, 'boolean_operator': None},

  ]),

  #################### Female #########################

   ("F 16-24", [       {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 16, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 24, 'boolean_operator': None}
                ]),

  ("F 25-34", [ {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 25, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 34, 'boolean_operator': None}
                ]),

   ("F 35-44", [
       {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 35, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 44, 'boolean_operator': None}
  ]),

                  ("F 45-54", [ {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 45, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '<=', 'value': 54, 'boolean_operator': None}
  ]),

      ("F 55-64", [ {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '>=', 'value': 55, 'boolean_operator': '&'},
                    {'column': 'age', 'operator': '<=', 'value': 64, 'boolean_operator': None}
  ]),

  ("F 65+", [
      {'column': 'Sex', 'operator': '==', 'value': 1, 'boolean_operator': '&'},
      {'column': 'age', 'operator': '>=', 'value': 65, 'boolean_operator': None},

  ])
            ]

all_conditions = [overall_group, subgroup_list_sex, subgroup_list_age, subgroup_list_simd] #, subgroup_list_age_sex]

In [ ]:
dem_group_dict = {"Overall": overall_group,
                  "Sex": subgroup_list_sex,
                  "Age": subgroup_list_age,
                  "SIMD": subgroup_list_simd}

In [ ]:
scenario_list = [f"Scenario_{i}" for i in range(1,36)]

In [ ]:
save_path = results_path / 'Tables/'
save_path.mkdir(parents=True, exist_ok=True)

for file_name, dem_group in dem_group_dict.items():
  results_table(
                dem_group=dem_group,
                nutrient_label_dict=nutrient_label_dict,
                error_columns=error_columns,
                scenario_list=scenario_list,
                scenario_label_map=scenario_label_map,
                file_name=file_name,
                save_path=save_path,
                results_path=results_path
                )